# 0.0 Notebook 00 - EDA and Cleaning the Raw Data

This is the first step of the pipeline. It reads three months of NYC Yellow Taxi trips (January to March 2024) from HDFS, checks the quality and the native precision of the columns, applies the cleaning rules and writes one clean base back to HDFS. Every storage format and precision level in the next notebooks is built from this base, so all experiments use exactly the same trips.

### 0.1 Imports

In [8]:
from pyspark.sql import SparkSession, functions as F

### 0.2 Data Source and Licence

The trip records are published by the NYC Taxi and Limousine Commission (TLC) as monthly Parquet files, together with the zone lookup table `taxi_zone_lookup.csv`The files were downloaded on 29 September 2026. 

There is no named licence. New York City's Open Data Law allows unrestricted use, as long as whoever redistributes the data names the source and the version and describes the changes; the changes made here are the cleaning rules in Step 5. The data is provided "as is", and there are no driver, vehicle or passenger identifiers, so the dataset has no personal data.

## SETUP & ENVORMENT

The data is read using PySpark from HDFS running in pseudo-distributed mode, which is the processing environment used throughout the project. The Spark session utilizes 4 local cores and 4 GB of memory for the driver, representing the virtual machine's limit.

The session timezone is set to UTC. Pickup and dropoff times are stored as timezone-naive timestamps; if Spark were to use the VM's local timezone (Europe/Dublin), the daylight saving time transition on March 31, 2024, could shift the calculated duration of some trips by one hour.

### 0.3- Initialazing Spark Session and Raw Files

The data is read with PySpark from HDFS running in pseudo-distributed mode. The session uses the 4 local cores and 4 GB for the driver.

The session time zone is set to UTC. Pickup and dropoff are stored as timestamps without a time zone, and with the VM's local zone (Europe/Dublin) the clock change on 31 March 2024 could shift some durations by one hour.

In [5]:
from pyspark.sql import SparkSession, functions as F

# Initialize the spark and give it power to compute 
spark = (SparkSession.builder.master("local[4]").appName("ca1-00-inspect")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())

# print results and path. 
df = spark.read.parquet("hdfs://localhost:9000/ca1/raw/yellow_tripdata_2024-*.parquet")
df.printSchema()
print("Quantity of lines:", df.count())

26/10/06 16:05:09 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://localhost:9000/ca1/raw/yellow_tripdata_2024-*.parquet.
java.net.ConnectException: Call From BDSP/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
	at org.apache.hadoop.util.dynamic.DynConstructors$Ctor.newInstanceChecked(DynConstructors.jav

Py4JJavaError: An error occurred while calling o50.parquet.
: java.net.ConnectException: Call From BDSP/127.0.1.1 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
	at java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
	at org.apache.hadoop.util.dynamic.DynConstructors$Ctor.newInstanceChecked(DynConstructors.java:58)
	at org.apache.hadoop.util.dynamic.DynConstructors$Ctor.newInstance(DynConstructors.java:70)
	at org.apache.hadoop.net.

The three monthly files share the same schema, so one path with a wildcard reads all of them. Together they have 9,554,778 trips and 19 columns. Pickup and dropoff come as `timestamp_ntz`, distance and fares as `double`, and the location IDs as `integer`.

### 0.3 - Quality Checks

Before cleaning anything, I checked the date range, how many trips have zero or negative distance or duration (most knowing as null_count).I also checked how many last more than 4 hours,and the median and 99th percentile of the distance.

In [7]:
# Comment this code for the future 
dur = F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")

df.select(F.min("tpep_pickup_datetime").alias("pickup_min"),
          F.max("tpep_pickup_datetime").alias("pickup_max"),
          F.avg((F.col("trip_distance") <= 0).cast("int")).alias("dist_zero_ou_neg"),
          F.avg((dur <= 0).cast("int")).alias("dur_zero_ou_neg"),
          F.avg((dur > 4 * 3600).cast("int")).alias("dur_maior_4h"),
          F.expr("percentile_approx(trip_distance, array(0.5, 0.99))").alias("dist_p50_p99")
         ).show(vertical=True, truncate=False)

NameError: name 'df' is not defined

It was found that 2.26% of the trips have zero or negative distance, 0.03% have zero or negative duration and 0.06% last more than 4 hours.  Also the median distance is 1.7 miles and the 99th percentile is 19.87 miles, so most trips are considere short.

### 0.4 Missing Values, Find Outliers Distances and Unknown Zones

This cell measures the share of nulls in the columns that could be features, the largest distance in the data and how many trips start or end in the two unknown zones (which can be called 264 and 265 in the zone lookup table).

In [17]:
# Distancia media - Medicao de numeros nulos 

cols = ["passenger_count", "RatecodeID", "PULocationID", "DOLocationID", "trip_distance"]
df.select([F.avg(F.col(c).isNull().cast("int")).alias(c) for c in cols]).show(vertical=True)

df.select(F.max("trip_distance").alias("dist_max"),
          F.avg(F.col("PULocationID").isin(264, 265).cast("int")).alias("pu_desconhecido"),
          F.avg(F.col("DOLocationID").isin(264, 265).cast("int")).alias("do_desconhecido")
         ).show(vertical=True)

zones = spark.read.option("header", True).csv("hdfs://localhost:9000/ca1/raw/taxi_zone_lookup.csv")
zones.filter(F.col("LocationID").cast("int") >= 264).show(truncate=False)

-RECORD 0------------------------------
 passenger_count | 0.07870010166641235 
 RatecodeID      | 0.07870010166641235 
 PULocationID    | 0.0                 
 DOLocationID    | 0.0                 
 trip_distance   | 0.0                 

-RECORD 0-------------------------------
 dist_max        | 312722.3             
 pu_desconhecido | 0.003847289806210... 
 do_desconhecido | 0.008883408908087661 

+----------+-------+--------------+------------+
|LocationID|Borough|Zone          |service_zone|
+----------+-------+--------------+------------+
|264       |Unknown|N/A           |N/A         |
|265       |N/A    |Outside of NYC|N/A         |
+----------+-------+--------------+------------+



it was found that `passenger_count` and `RatecodeID` are null in 7.87% of the trips, the largest distance is 312,722.3 miles, which is consider a outlier. Also 0.38% of pickups and 0.89% of dropoffs are in unknown zones.

The features chosen for the network are distance, hour of day and day of week from the pickup, and the pickup and dropoff zones. `passenger_count` and `RatecodeID` are not used, so no imputation is needed.

### 0.5 - Cleaning Data

The rules are applied in sequence, on the original values, and each line shows how many trips it has been removed:

* Pickup between 1 January and 31 March 2024
* Distance above 0 and up to 100 miles (the 99th percentile is 19.87 miles, so 100 keeps long airport trips and removes typing errors)
* Duration between 60 seconds and 4 hours
* Speed up to 80 mph, which removes trips that are physically impossible in New York
* Known pickup and dropoff zones.

In [18]:
dur = F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
t_ini = F.lit("2024-01-01 00:00:00").cast("timestamp_ntz")
t_fim = F.lit("2024-04-01 00:00:00").cast("timestamp_ntz")

filtros = [
    ("periodo jan-mar 2024", (F.col("tpep_pickup_datetime") >= t_ini) & (F.col("tpep_pickup_datetime") < t_fim)),
    ("distancia (0, 100] mi", (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100)),
    ("duracao [60 s, 4 h]", (dur >= 60) & (dur <= 4 * 3600)),
    ("velocidade <= 80 mph", F.col("trip_distance") / (dur / 3600) <= 80),
    ("zonas conhecidas", ~F.col("PULocationID").isin(264, 265) & ~F.col("DOLocationID").isin(264, 265)),
]

atual = df
n_ant = df.count()
print(f"{'inicio':<25}{n_ant:>12,}")
for nome, cond in filtros:
    atual = atual.filter(cond)
    n = atual.count()
    print(f"{nome:<25}{n:>12,}   removidas: {n_ant - n:>9,} ({(n_ant - n) / n_ant:.2%})")
    n_ant = n

inicio                      9,554,778
periodo jan-mar 2024        9,554,757   removidas:        21 (0.00%)
distancia (0, 100] mi       9,338,741   removidas:   216,016 (2.26%)
duracao [60 s, 4 h]         9,298,539   removidas:    40,202 (0.43%)
velocidade <= 80 mph        9,298,271   removidas:       268 (0.00%)
zonas conhecidas            9,213,529   removidas:    84,742 (0.91%)


The cleaning process was completed and it left 9,213,529 trips and removes 3.57% of the raw data. 

The filters that remove the most are distance (216,016 trips) and unknown zones (84,742).

### 0.6 - Saving the Clean Data into HDFS. 

In [19]:
dur = F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
clean = atual.withColumn("duration_s_true", dur)
clean.write.mode("overwrite").parquet("hdfs://localhost:9000/ca1/clean/yellow_2024q1")

chk = spark.read.parquet("hdfs://localhost:9000/ca1/clean/yellow_2024q1")
print("linhas:", chk.count())
chk.select(F.min("duration_s_true").alias("dur_min"),
           F.max("duration_s_true").alias("dur_max"),
           F.min("tpep_pickup_datetime").alias("pickup_min"),
           F.max("tpep_pickup_datetime").alias("pickup_max")).show(vertical=True)

linhas: 9213529
-RECORD 0-------------------------
 dur_min    | 60                  
 dur_max    | 14344               
 pickup_min | 2024-01-01 00:00:00 
 pickup_max | 2024-03-31 23:59:59 



The clean base has 9,213,529 trips, with durations between 60 and 14,344 seconds and every pickup in the year of 2024. The 3.57% removed are probably data that doesnt express real real trips, thefore, it does not affect future analisys.